# Demo: cosa distingue gli schemi di uno studente da quelli generati da un'AI

**Tirocinio (Informatica UniVR):** dai miei schemi di studio estraggo un *vocabolario* di regole (cosa fare, cosa non fare, stile, numeri misurabili) da dare a un modello (Claude/GPT) perché generi schemi nuovi nel mio stile.
Questa demo misura i numeri che separano i miei schemi dalle pagine generate e prepara lo zip col vocabolario da dare al modello.

Gira tutto qui, **senza chiavi, credenziali o Google Drive**: solo codice (niente modelli, niente OCR), dati di esempio inclusi nel repo (schemi dello studente + pagine generate da Claude; nessuna pagina del docente).

Esegui le celle in ordine (*Runtime > Esegui tutto*). Circa 2-3 minuti in tutto.

## 1. Scarica il repo e installa

In [ ]:
!git clone -q https://github.com/03gilbe-design/vocabolario-schemi-studio.git
%cd vocabolario-schemi-studio/strumenti/3_misura_stile_dai_numeri
!pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium > /dev/null 2>&1 && echo chromium pronto

## 2. Trova i numeri che distinguono lo studente dall'AI
Confronta 7 pagine di appunti dello studente (PDF) con 7 pagine generate da Claude (HTML/SVG). Scrive in `risultati_numeri/` la classifica, le frasi da dare al modello e i range per il controllo.

In [ ]:
!python -X utf8 trova_differenze.py --suoi esempi_numeri/suoi/Architettura_Gilberto.pdf --generate esempi_numeri/generate --prof-parole 385

from IPython.display import Image, Markdown, display
display(Image('risultati_numeri/classifica.png'))
display(Markdown(open('risultati_numeri/FRASI_PER_IL_MODELLO.md', encoding='utf-8').read()))

## 3. Controlla una pagina generata: OK o FUORI
Rende in PNG una pagina di esempio (Claude con le regole, prova r1) e la confronta con i range dello studente. Ogni FUORI ha il consiglio per correggerla.

**Come leggere il risultato:** questa pagina passa i **3 numeri del colore** (colore forte, zone di colore forte, tinta principale): è l'unica su 18 pagine generate a farlo. Gli altri FUORI (testo piccolo, troppe pillole, fondo vuoto...) **non sono un errore della demo**: sono il lavoro ancora da fare sulle regole.

In [ ]:
src = 'esempi_numeri/generate/claude_CON_REGOLE_r1/pagina.html'
# rendi() usa Playwright sincrono: dentro Colab va lanciato in un processo a parte
!python -X utf8 -c "from pathlib import Path; from trova_differenze import rendi; rendi(Path('{src}'), Path('pagina.png'))"
display(Image('pagina.png', width=400))
!python -X utf8 controlla_stile.py pagina.png --sorgente {src} --rif risultati_numeri/numeri.json --prof-parole 385

## 4. (Opzionale) Con i tuoi file
Carica **un PDF dei tuoi schemi** e **almeno 3 pagine generate da un'AI** (`.html`, `.svg` o `.png`). I file restano in questa sessione Colab.
Non hai pagine generate? In chat allega la pagina del prof e incolla `per_la_chat/PROMPT_SENZA_REGOLE.txt` del repo, una chat nuova per pagina.
Se un file è sbagliato (vuoto, Word, scansione bianca) lo script si ferma con un messaggio `ERRORE:` che dice cosa fare.

In [ ]:
USA_I_TUOI_FILE = False #@param {type:"boolean"}
#@markdown Spunta e riesegui questa cella per caricare i tuoi file.
if not USA_I_TUOI_FILE:
    print('Saltata: uso gli esempi del repo. Spunta USA_I_TUOI_FILE per caricare i tuoi.')
else:
    from google.colab import files
    import shutil, os
    print('1) i tuoi schemi (PDF o PNG)'); suoi = list(files.upload())
    print('2) le pagine generate (.html/.svg/.png)'); gen = list(files.upload())
    os.makedirs('miei/generate', exist_ok=True)
    for f in suoi: shutil.move(f, 'miei/' + f)
    for f in gen: shutil.move(f, 'miei/generate/' + f)
    args = ' '.join(f'"miei/{f}"' for f in suoi)
    !python -X utf8 trova_differenze.py --suoi {args} --generate miei/generate --out risultati_miei
    if os.path.exists('risultati_miei/numeri.json'):  # se c'e' un ERRORE: sopra, lo script si e' fermato
        display(Image('risultati_miei/classifica.png'))
        display(Markdown(open('risultati_miei/FRASI_PER_IL_MODELLO.md', encoding='utf-8').read()))

## 5. Scarica i file da dare al modello
Crea `per_la_chat.zip` con i 3 file della ricetta del README: `PACCHETTO_MODELLO.md`, `FRASI_PER_IL_MODELLO.md` (le tue se hai fatto il punto 4, se no quelle di `per_la_chat/`) e `PROMPT.txt`.
In chat (ChatGPT, Claude, Gemini): allega `PACCHETTO_MODELLO.md` e `FRASI_PER_IL_MODELLO.md` + la pagina del prof, incolla il testo di `PROMPT.txt`. Lo zip va scompattato: non tutte le chat aprono i .zip.
Senza Colab: gli stessi file sono già pronti nella cartella `per_la_chat/` del repo.

In [ ]:
import zipfile, os
from pathlib import Path
from google.colab import files
chat = Path('../../per_la_chat')
frasi = Path('risultati_miei/FRASI_PER_IL_MODELLO.md')
if not frasi.exists(): frasi = chat / 'FRASI_PER_IL_MODELLO.md'
with zipfile.ZipFile('per_la_chat.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(chat / 'PACCHETTO_MODELLO.md', 'PACCHETTO_MODELLO.md')
    z.write(frasi, 'FRASI_PER_IL_MODELLO.md')
    z.write(chat / 'PROMPT.txt', 'PROMPT.txt')
print('per_la_chat.zip:', os.path.getsize('per_la_chat.zip') // 1024, 'KB, frasi da', frasi)
files.download('per_la_chat.zip')
